<!-- nav -->
[← 3 · The smallest expression, a literal](03_The_Smallest_Expression_A_Literal.ipynb) · [Home](../../README.md) · [5 · Operators →](05_Operators.ipynb)

# 4 · Names

After a literal, the next smallest expression is a name: `x`, a value the expression doesn't hold but finds in its
**scope** when it is evaluated. This notebook models a name in each language, asks a term which names it needs, and
shows that a name is kept exactly as written, even one its language can't spell.

The code starts from `dialects()`, the toolkit's store with every dialect installed and defined. `S` is its
holders' namespace, `store.Expressions`, and `amd64` the ABI dialect C's and C++'s holders name, as in tutorial 3.

In [1]:
import assert from "node:assert/strict";
import { Evaluators, Expressions as E } from "@mbse/expressions/Patterns/Expressions";
import * as C from "@mbse/expressions/C/Expressions";
import * as Cpp from "@mbse/expressions/Cpp/Expressions";
import * as Excel from "@mbse/expressions/Excel/Expressions";
import { Symbolics, Terms } from "@mbse/expressions/Framework";
import * as Latex from "@mbse/expressions/LaTeX/Expressions";
import * as Matlab from "@mbse/expressions/MATLAB/Expressions";
import * as Python3 from "@mbse/expressions/Python/Python3";
import * as SystemVerilog from "@mbse/expressions/SystemVerilog/Expressions";
import * as TypeScript5 from "@mbse/expressions/TypeScript/TypeScript5";
import * as Translators from "@mbse/expressions/Translators";
import { dialects, show } from "./toolkit.js";

const store = dialects();
const [S, Term] = [store.Expressions, store.Expressions.Framework.Term];
const amd64 = store.Instances.Expressions.SystemV.AMD64;

<a id="T04.1"></a>
## 04.1 · A variable

Patterns' `x` is a term of kind `variable`, whose one attribute is its `name`. It holds no value: evaluation looks the
name up in the scope it's given, here `{"x": 42}`:

In [2]:
const x = store.transaction(() => S.Patterns.Expressions.Expression((e: any) => e.root(
  (u: any) => u.variable((v: any) => v.name("x")))));
const form = Terms.form(Terms.term(x));
console.log(form.kind, form.attributes.get("name"), show(x));
console.log(String(Evaluators.OfAny(x, { x: 42n })));

variable x x


42


Given no scope that binds `x`, evaluation raises. Validation finds the same problem first, without evaluating:
`validate` takes the names the caller will bind, and reports every other name the term reads.

In [3]:
try {
  Evaluators.OfAny(x, {});
} catch (error) {
  console.log(String(error));
}
console.log(JSON.stringify(E.DIALECT.validate(x)));
console.log(JSON.stringify(E.DIALECT.validate(x, ["x"])));

KeyError: variable 'x' is not bound


["variable 'x' is not bound"]


[]


<a id="T04.2"></a>
## 04.2 · A name in each language

Each language's dialect has its own kind for a name, each printed `x`:

- **Python3**'s `variable` (`ast`'s `Name`), and **C**'s, **Excel**'s and **MATLAB**'s: each the framework's `variable`,
  the kind Patterns' is too.
- **TypeScript5**'s `identifier`, as TypeScript's own syntax tree calls it.
- **C++**'s `idexpression` and **SystemVerilog**'s `nameexpression`, which have no `variable`
  ([CAP-230: the C++ expression dialect](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-230.md), [CAP-231: the SystemVerilog expression dialect, complete](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-231.md)): each holds a **name**, a helper of its own, since a name in those languages need not be a single
  identifier (`std::cout`, `f<int>`, `p::x`, `$unit::x`). Here it is the simplest, an `identifier`, its `spelling`
  `x`.
- **Excel**'s `variable` (a `LET` name or a defined name) beside its `cell`, `A1`, a reference no scope binds: the
  workbook resolves it.

Each model is built in its dialect's holder, and its dialect's `Text.ToText` gives the language's own text back.

Python's notebook also reads Python's text, with `Python3.Text.FromText`, into the same term; TypeScript builds it, as above.

In [4]:
const identifier = (spelling: string) => (n: any) => n.identifier((i: any) => i.spelling(spelling)); // a C++ or SystemVerilog name
const names: [any, any][] = store.transaction(() => [
  [Python3, S.Python.Python3.Expression((e: any) => e.root((u: any) => u.variable((v: any) => v.name("x"))))],
  [TypeScript5, S.TypeScript.TypeScript5.Expression((e: any) => e.root((u: any) => u.identifier((i: any) => i.name("x"))))],
  [C, S.C.Expressions.Expression((e: any) => e.root((u: any) => u.variable((v: any) => v.name("x"))).abi([amd64]))],
  [Cpp, S.Cpp.Expressions.Expression((e: any) => e.root(
    (u: any) => u.idexpression((r: any) => r.name(identifier("x")))).abi([amd64]))],
  [SystemVerilog, S.SystemVerilog.Expressions.Expression((e: any) => e.root(
    (u: any) => u.nameexpression((r: any) => r.name(identifier("x")))))],
  [Excel, S.Excel.Expressions.Expression((e: any) => e.root((u: any) => u.variable((v: any) => v.name("x"))))],
  [Excel, S.Excel.Expressions.Expression((e: any) => e.root((u: any) => u.cell((c: any) => c.address("A1"))))],
  [Matlab, S.MATLAB.Expressions.Expression((e: any) => e.root((u: any) => u.variable((v: any) => v.name("x"))))],
]);
for (const [dialect, name] of names) {
  console.log(dialect.Expressions.DIALECT.name().padEnd(13), Terms.form(Terms.term(name)).kind.padEnd(14),
    dialect.Text.ToText(name));
}
assert.deepEqual(names.map(([dialect, name]) => dialect.Text.ToText(name)), ["x", "x", "x", "x", "x", "=x", "=A1", "x"]);

Python3       variable       x


TypeScript5   identifier     x


C             variable       x


Cpp           idexpression   x


SystemVerilog nameexpression x


Excel         variable       =x


Excel         cell           =A1


MATLAB        variable       x


A C++ or SystemVerilog name is the term's `name` attribute, itself a part with a kind and attributes; `Terms.nameOf`
reads the spelling the term refers by, whatever kind holds it:

In [5]:
for (const [dialect, name] of names.slice(3, 5)) {
  const form = Terms.form(Terms.term(name));
  const part: any = form.attributes.get("name");
  console.log(dialect.Expressions.DIALECT.name().padEnd(13), form.kind, "holds", part.kind,
    `'${part.attributes.get("spelling")}'`);
}
console.log(names.map(([, name]) => Terms.nameOf(Terms.term(name)) ?? "-").join(" "));

Cpp           idexpression holds identifier 'x'


SystemVerilog nameexpression holds identifier 'x'


x x x x x x - x


The writers tutorial 3 showed write the same terms as these builder calls: Patterns' `variable`, Python3's `name` and
C's `identifier`, which writes C's `variable`:

In [6]:
const written = store.transaction(() => [
  Term(E.variable("x")), Term(Python3.Expressions.name("x")), Term(C.Expressions.identifier("x"))]);
console.log(JSON.stringify(written.map((a: any, i: number) => Terms.same(a, [x, names[0]![1], names[2]![1]][i]))));

[true,true,true]


<a id="T04.3"></a>
## 04.3 · Free names

`Symbolics.free(term, DIALECT)` is what a term needs from its scope: the names it reads that nothing within it binds,
read as its dialect reads them. Python's builtins are always bound, so `len(x)` needs `x` alone, and validation reports
`x`, not `len`:

In [7]:
const length = store.transaction(() => S.Python.Python3.Expression((e: any) => e.root((u: any) => u.call((c: any) => c
  .func((f: any) => f.variable((v: any) => v.name("len"))).args([(a: any) => a.variable((v: any) => v.name("x"))])))));
assert.equal(Python3.Text.ToText(length), "len(x)");
console.log(Python3.Text.ToText(length), "needs", [...Symbolics.free(length, Python3.Expressions.DIALECT)].sort().join(", "));
console.log(JSON.stringify(Python3.Expressions.DIALECT.validate(length)));

len(x) needs x


["argument 0: variable 'x' is not bound"]


TypeScript's `undefined` is not a keyword but an `identifier`, as TypeScript itself parses it: a global the scope
provides, so a term that reads it needs it:

In [8]:
const undefined_ = store.transaction(() => S.TypeScript.TypeScript5.Expression((e: any) => e.root(
  (u: any) => u.identifier((i: any) => i.name("undefined")))));
console.log(Terms.form(Terms.term(undefined_)).kind, TypeScript5.Text.ToText(undefined_), "needs",
  [...Symbolics.free(undefined_, TypeScript5.Expressions.DIALECT)].sort().join(", "));

identifier undefined needs undefined


<a id="T04.4"></a>
## 04.4 · A name kept as written

`max-temp` is no name in Python: Python would read it as `max - temp`. Built as a name, it is kept in the tree as
written, printed as written, and reported by validation, never renamed ([PRI-002: preserve, don't lose](https://github.com/pitaman71/mbse-basis/blob/main/docs/PRINCIPLES.md#PRI-002)):

In [9]:
const max_temp = store.transaction(() => S.Python.Python3.Expression((e: any) => e.root(
  (u: any) => u.variable((v: any) => v.name("max-temp")))));
console.log(Terms.form(Terms.term(max_temp)).attributes.get("name"), Python3.Text.ToText(max_temp));
console.log(JSON.stringify(Python3.Expressions.DIALECT.validate(max_temp, ["max-temp"])));

max-temp max-temp


["a name must be an identifier, got 'max-temp'"]


Each language that checks its names reports one it can't spell, in its own words; none renames it:

In [10]:
const unspellable: [any, any][] = store.transaction(() => [
  [TypeScript5, S.TypeScript.TypeScript5.Expression(
    (e: any) => e.root((u: any) => u.identifier((i: any) => i.name("max-temp"))))],
  [C, S.C.Expressions.Expression((e: any) => e.root((u: any) => u.variable((v: any) => v.name("max-temp"))).abi([amd64]))],
  [Cpp, S.Cpp.Expressions.Expression((e: any) => e.root(
    (u: any) => u.idexpression((r: any) => r.name(identifier("max-temp")))).abi([amd64]))],
  [SystemVerilog, S.SystemVerilog.Expressions.Expression((e: any) => e.root(
    (u: any) => u.nameexpression((r: any) => r.name(identifier("max-temp")))))],
]);
for (const [dialect, name] of unspellable) {
  console.log(dialect.Expressions.DIALECT.name().padEnd(13), dialect.Text.ToText(name).padEnd(9),
    JSON.stringify(dialect.Expressions.DIALECT.validate(name, ["max-temp"])));
}

TypeScript5   max-temp  ["an identifier's name cannot be 'max-temp'"]


C             max-temp  ["'max-temp' is not an identifier"]


Cpp           max-temp  ["'max-temp' is not an identifier"]


SystemVerilog max-temp  ["'max-temp' is not an identifier"]


<a id="T04.5"></a>
## 04.5 · Names across languages

A translator ([CAP-101: translation between dialects](https://github.com/pitaman71/mbse-basis/blob/main/docs/capabilities/CAP-101.md)) writes `x` as each language's own name, the same term 04.2 built, and translates it back to
Patterns' `variable`. TypeScript5 has no translator yet, and says so:

In [11]:
const built = new Map([...names].reverse().map(([dialect, name]) => [dialect.Expressions.DIALECT.name(), name])); // Excel's variable, not its cell
for (const dialect of [Python3, TypeScript5, C, Cpp, SystemVerilog, Excel, Matlab, Latex] as any[]) {
  const target = dialect.Expressions.DIALECT;
  let there: any, back: any;
  try {
    [there, back] = store.transaction(() => {
      const there = Term(Translators.between(E.DIALECT, target).forward(x));
      return [there, Term(Translators.between(target, E.DIALECT).forward(there))];
    });
  } catch (error) {
    console.log(target.name().padEnd(13), String(error));
    continue;
  }
  const same = built.has(target.name()) ? String(Terms.same(there, built.get(target.name()))) : "-";
  console.log(target.name().padEnd(13), dialect.Text.ToText(there).padEnd(3), Terms.form(there).kind.padEnd(14),
    same.padEnd(5), "back:", show(back), Terms.same(back, x));
}

Python3       x   variable       true  back: x true


TypeScript5   LookupError: no translator between Patterns and TypeScript5


C             x   variable       true  back: x true


Cpp           x   idexpression   true  back: x true


SystemVerilog x   nameexpression true  back: x true


Excel         =x  variable       true  back: x true


MATLAB        x   variable       true  back: x true


LaTeX         x   variable       -     back: x true


## Why it works this way

- **A name is a reference, not a value.** A term that reads `x` holds the name, and the scope gives its value, so the
  same constraint is evaluated against any data that binds its names, and `free` says what that data must bind.
- **Each language's own kind.** A dialect models its language's syntax tree as that language has it: TypeScript's
  `identifier`, C++'s `idexpression`, SystemVerilog's `nameexpression`, so that its text prints back as written.
- **Kept as written, reported, never repaired.** A name that a language can't spell is kept, printed and reported, so
  nothing in a model is lost or silently changed on its way through a tool.

<!-- nav -->
[← 3 · The smallest expression, a literal](03_The_Smallest_Expression_A_Literal.ipynb) · [Home](../../README.md) · [5 · Operators →](05_Operators.ipynb)